In [ ]:
import gymnasium as gym
import tensorflow as tf
import numpy as np
from keras.models import Model, clone_model
from keras.layers import Dense, Input
from keras.losses import MeanSquaredError
import random
from collections import deque

env = gym.make("CartPole-v1")





In [ ]:

# ==========================================
# Q-Network & Target Network Setup
# ==========================================
net_input = Input(shape=(4,))


x = Dense(64, activation='relu')(net_input)
x = Dense(32, activation='relu')(x)
net_output = Dense(2, activation='linear')(x)


q_net = Model(inputs=net_input, outputs=net_output)
q_net.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))
loss_fn = MeanSquaredError()

target_net = clone_model(q_net)
target_net.set_weights(q_net.get_weights())


In [ ]:

# ==========================================
# Parameters
# ==========================================
EPSILON = 1.0          
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995  
GAMMA = 0.95
NUM_EPISODES = 500
BATCH_SIZE = 64         # Batch size thora bara kiya hai achi learning ke liye
UPDATE_FREQ = 4         # Har 4 step baad network update hoga (Speed boost)



memory = deque(maxlen=10000)


In [9]:

# ==========================================
# ⚡ COMPILED FAST TRAINING STEP ⚡
# @tf.function isko C++ graph mein compile kar dega (Super Fast)
# ==========================================
@tf.function
def train_step(states, actions, targets):
    with tf.GradientTape() as tape:
        q_values = q_net(states)
        masks = tf.one_hot(actions, 2)
        q_action = tf.reduce_sum(tf.multiply(q_values, masks), axis=1)
        loss = loss_fn(targets, q_action)

    grads = tape.gradient(loss, q_net.trainable_variables)
    q_net.optimizer.apply_gradients(zip(grads, q_net.trainable_variables))

def policy(state, explore=0.0):
    if tf.random.uniform(shape=(), maxval=1.0) <= explore:
        return np.random.randint(0, 2)
    q_values = q_net(state)
    return int(tf.argmax(q_values[0]).numpy())


In [ ]:

# ==========================================
# Training Loop
# ==========================================
step_count = 0

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    state = state.reshape(1, 4)
    done = False
    total_reward = 0
    
    
    

    while not done:
        action = policy(state, EPSILON)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = next_state.reshape(1, 4)

        memory.append((state, action, reward, next_state, done))
        
        state = next_state
        total_reward += reward
        step_count += 1
        
        

        # Sirf har 4th step par train karein (Computation bachegi aur speed barhegi)
        if len(memory) >= BATCH_SIZE and step_count % UPDATE_FREQ == 0:
            minibatch = random.sample(memory, BATCH_SIZE)
            
            batch_states = np.vstack([x[0] for x in minibatch])
            batch_actions = np.array([x[1] for x in minibatch], dtype=np.int32)
            batch_rewards = np.array([x[2] for x in minibatch], dtype=np.float32)
            batch_next_states = np.vstack([x[3] for x in minibatch])
            batch_dones = np.array([x[4] for x in minibatch], dtype=np.float32)
            
            target_q_values = target_net(batch_next_states).numpy()
            max_target_q_values = np.max(target_q_values, axis=1)
            target_values = batch_rewards + (GAMMA * max_target_q_values * (1.0 - batch_dones))

            # Naya aur fast compiled function call karein
            train_step(
                tf.convert_to_tensor(batch_states, dtype=tf.float32),
                tf.convert_to_tensor(batch_actions, dtype=tf.int32),
                tf.convert_to_tensor(target_values, dtype=tf.float32)
            )

    target_net.set_weights(q_net.get_weights())

    if EPSILON > EPSILON_MIN:
        EPSILON *= EPSILON_DECAY

    print(f"Episode {episode + 1}: Total Reward: {total_reward}, Epsilon: {EPSILON:.4f}")

env.close()

Episode 1: Total Reward: 23.0, Epsilon: 0.9950
Episode 2: Total Reward: 19.0, Epsilon: 0.9900
Episode 3: Total Reward: 14.0, Epsilon: 0.9851
Episode 4: Total Reward: 22.0, Epsilon: 0.9801
Episode 5: Total Reward: 22.0, Epsilon: 0.9752
Episode 6: Total Reward: 24.0, Epsilon: 0.9704
Episode 7: Total Reward: 28.0, Epsilon: 0.9655
Episode 8: Total Reward: 13.0, Epsilon: 0.9607
Episode 9: Total Reward: 63.0, Epsilon: 0.9559
Episode 10: Total Reward: 47.0, Epsilon: 0.9511
Episode 11: Total Reward: 21.0, Epsilon: 0.9464
Episode 12: Total Reward: 20.0, Epsilon: 0.9416
Episode 13: Total Reward: 12.0, Epsilon: 0.9369
Episode 14: Total Reward: 12.0, Epsilon: 0.9322
Episode 15: Total Reward: 14.0, Epsilon: 0.9276
Episode 16: Total Reward: 18.0, Epsilon: 0.9229
Episode 17: Total Reward: 10.0, Epsilon: 0.9183
Episode 18: Total Reward: 12.0, Epsilon: 0.9137
Episode 19: Total Reward: 20.0, Epsilon: 0.9092
Episode 20: Total Reward: 23.0, Epsilon: 0.9046
Episode 21: Total Reward: 10.0, Epsilon: 0.9001
E

In [ ]:
q_net.save("dqn_cartpole_model.h5")

print("Model saved as dqn_cartpole_model.h5")

Model saved as dqn_cartpole_model.h5
